# 강의 04 · 실습 2 — 체크포인트와 사람 개입 · (4) 고난도 I — 조건부 개입

## 1. 문제상황

- 여행사 고객센터의 항공권 변경 문의는 하루 수십 통이고, 담당자는 한 사람입니다.
- 지금의 처리 흐름은 모든 문의에서 담당자 방침을 기다리므로, 담당자가 자리를 비우면 3만 원짜리 소액 변경 문의까지 전부 멈춰 있습니다.
- 담당자가 정말 봐야 하는 문의는 환불·변경 금액이 큰 건뿐이고, 소액 건은 규정 수수료를 안내하는 정해진 방침으로 충분합니다.
- 같은 고객이 소액 문의와 고액 문의를 번갈아 보내기도 하므로, 어느 쪽이든 앞선 대화는 이어져야 합니다.

## 2. 문제와 목표

- **문제**: 소액 문의까지 담당자 방침을 기다리므로 담당자가 없으면 전부 멈추고, 담당자는 봐야 할 건과 보지 않아도 되는 건을 구분하지 못한 채 모든 건을 봅니다.
- **목표**
  - 고객 메일에 딸린 변경 금액을 보고, 기준 금액 이상이면 담당자에게 방침을 묻고(멈춤), 기준 미만이면 정해진 기본 방침을 자동으로 채웁니다(멈추지 않음).
    - 기준 금액과 기본 방침 문장: 「4. 단계별 요구사항」에 값이 있습니다.
  - 두 경우 모두 그 방침으로 초안을 써서 발송하는 처리 흐름을 만듭니다.
  - 대화 기록과 체크포인트는 같은 `thread_id` 아래 그대로 유지합니다.
- **목표 달성 여부의 판정 기준**: 같은 `thread_id`로 고액 문의 한 통과 소액 문의 한 통을 차례로 넣었을 때,
  - 고액 문의는 초안을 만들기 전에 멈춰 담당자 방침을 받은 뒤 발송되고, 소액 문의는 멈추지 않고 기본 방침이 채워진 채 곧바로 발송되며,
  - 두 통 모두 초안에 방침의 핵심이 들어 있고, 두 번째 통의 대화 기록에 첫 통의 메일과 답장이 남아 있는 것을 실행 결과에서 확인합니다.
  - 담당자의 방침은 대본으로 미리 넣습니다.

## 3. 워크플로우 다이어그램

![워크플로우 다이어그램](imgs/lec04_ex02_s4_diagram.svg)

## 4. 단계별 요구사항

1. **상태를 정의합니다.**
    - 대화 기록(`messages`, `add_messages` 리듀서), 변경 금액(`amount`, 정수), 담당자 방침(`policy`), 답장 초안(`draft`), 발송 여부(`sent`) 키 다섯 개를 가지는 상태를 선언합니다.
2. **컨텍스트 관리 노드를 만듭니다.**
    - manage 노드는 `messages`의 길이가 `KEEP`(이 실습에서는 4) 이하이면 아무것도 바꾸지 않습니다.
    - `KEEP`을 넘으면 최근 `KEEP` 개만 원문으로 남기고, 그 앞의 메시지들을 요약 지시문 `SUMMARY_RULE`로 모델에 요약시킨 뒤, 오래된 메시지들을 `RemoveMessage`로 지우고 요약 `SystemMessage` 하나를 넣습니다.
    - 요약 지시문은 「다음 여행사 고객센터 대화를 고객이 문의한 내용과 상담원이 안내한 처리 중심으로 두 문장 이내 한국어로 요약한다. 새 정보를 지어내지 않는다.」입니다.
3. **방침 노드 두 개를 만듭니다.**
    - ask_policy 노드는 `interrupt()`로 멈추고 질문·마지막 고객 메일(가장 최근의 `HumanMessage`)·변경 금액을 담당자에게 보낸 뒤, 담당자가 준 값을 `policy` 키에 씁니다.
    - auto_policy 노드는 멈추지 않고 정해진 기본 방침 문장(`DEFAULT_POLICY`)을 `policy` 키에 씁니다.
    - 기본 방침 문장은 「규정대로 변경 수수료 3만 원을 안내한다. 변경 희망 날짜의 좌석을 확인해 준다.」입니다.
4. **초안 노드와 발송 노드를 만듭니다.**
    - draft 노드는 `policy` 키의 방침을 시스템 프롬프트에 넣고 `messages` 전체를 모델에 넣어 두 문장짜리 초안을 `draft` 키에 씁니다.
    - send 노드는 초안을 발송하고(화면 출력), 보낸 답장을 `AIMessage`로 `messages` 키에 쌓고 `sent` 키에 `True`를 씁니다.
5. **그래프에 노드를 등록합니다.**
    - 다섯 노드(manage, ask_policy, auto_policy, draft, send)를 이름과 함께 등록합니다.
6. **엣지를 연결합니다.**
    - START → manage는 고정 엣지입니다.
    - manage 뒤에는 `amount`가 기준 금액(`THRESHOLD`, 이 실습에서는 20만 원 = 200000) 이상이면 `"ask_policy"`, 미만이면 `"auto_policy"`를 돌려주는 판단 함수로 조건부 엣지를 추가합니다.
    - ask_policy → draft, auto_policy → draft, draft → send, send → END는 고정 엣지입니다.
7. **체크포인터를 장착해 컴파일합니다.**
    - SQLite 파일에 저장하는 체크포인터(`SqliteSaver`)를 열어 `compile(checkpointer=…)`에 넘기고, `thread_id` 설정을 만듭니다.
    - `thread_id`는 `customer-8102`입니다.
8. **그래프를 실행합니다.**
    - 같은 `thread_id`로 고액 문의(금액 기준 이상)와 소액 문의(기준 미만)를 차례로 넣습니다. 입력에는 메일과 함께 변경 금액을 `amount` 키 값으로 넣습니다.
    - 고액 문의는 멈춘 지점과 담당자에게 간 내용을 출력한 뒤 `Command(resume=…)`으로 방침을 넘기고, 소액 문의는 `invoke` 반환값에 `__interrupt__`가 없음을 출력합니다.
    - 두 통 모두 방침·초안·대화 기록을 출력합니다.
    - 출력 줄에는 「[멈춤 여부]」「[멈춘 지점]」「[담당자에게 간 내용]」「[방침]」「[초안]」「[발송]」「[대화 기록]」 표지를 붙입니다.
    - 멈춘 지점은 `next = (노드 이름,)` 형태로 출력합니다.
    - 멈춤 여부는 `__interrupt__ 있음 = True` 또는 `False`로 출력합니다.
    - 값은 「6. 코드 — 스텝바이스텝」 단계 0에 주어져 있습니다.

## 5. 코드 골격 — LangGraph 5단

랭그래프(LangGraph)로 그래프를 세우는 순서는 다음 다섯 단계입니다. 조건부 엣지는 ④ 엣지 연결 단계에, 체크포인터와 `interrupt()`는 ⑤ 컴파일과 실행 단계에 속합니다.

| 단계 | 하는 일 | 사용하는 코드 | 대응하는 요구사항 |
|---|---|---|---|
| ① 상태 정의 | 노드들이 함께 읽고 쓸 키를 선언합니다 | `class PolicyState(TypedDict)`, `Annotated[list, add_messages]` | 1 |
| ② 노드 함수 정의 | 상태를 받아 바뀐 키만 돌려주는 함수를 만듭니다 | `def manage(state) -> dict`, `interrupt()` | 2, 3, 4 |
| ③ 그래프 빌더 생성과 노드 등록 | 빈 그래프를 열고 함수에 이름을 붙여 등록합니다 | `StateGraph(PolicyState)`, `add_node` | 5 |
| ④ 엣지 연결 | 노드 사이의 순서와 분기를 정합니다 | `add_edge`, `add_conditional_edges` | 6 |
| ⑤ 컴파일과 실행 | 체크포인터를 달아 컴파일하고, `thread_id`를 넘겨 실행하고, 멈춘 지점에서 이어 갑니다 | `compile(checkpointer=…)`, `invoke`, `Command(resume=…)`, `get_state` | 7, 8 |

## 6. 코드 — 스텝바이스텝

### 단계 0 — 준비

라이브러리를 불러오고 모델을 준비합니다. 체크포인트를 저장할 파일 위치도 여기서 정합니다.

- API 키는 `.env` 파일에서 읽습니다.
- `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 둡니다. `find_dotenv()`가 노트북 위치에서 상위 폴더로 올라가며 찾습니다.
- `.env` 파일에는 다음 한 줄만 넣습니다.

```
OPENAI_API_KEY=발급받은_키
```

- 체크포인트 파일은 실행할 때마다 새 임시 폴더에 만듭니다. 지난 실행의 저장 기록이 이번 실행에 섞이지 않게 하기 위해서입니다.
- 대화 기록 출력은 `show_messages(msgs)`로 합니다.

In [ ]:
import sqlite3
import tempfile
from pathlib import Path
import os

from dotenv import load_dotenv, find_dotenv
from typing import Annotated, TypedDict

from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, HumanMessage, RemoveMessage, SystemMessage
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.types import Command, interrupt

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("OPENAI_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 OPENAI_API_KEY 한 줄을 넣습니다.")

llm = init_chat_model("openai/gpt-5.6-luna", model_provider="litellm")

DB_PATH = Path(tempfile.mkdtemp(prefix="lec04_ex02_")) / "checkpoint.db"   # 체크포인트를 저장할 파일


def show_messages(msgs: list) -> None:
    """대화 기록을 메시지 종류와 앞부분 글자만 한 줄씩 출력한다."""
    for m in msgs:
        print(f"      [{type(m).__name__}] {str(m.content)[:90]}")


print("모델 준비를 마쳤습니다. 체크포인트 파일:", DB_PATH.name)


# 주어진 자료
EMAIL_BIG = "가족 네 명 항공권을 전부 다음 달로 미루고 싶습니다. 총 68만 원 결제했는데 변경 비용이 어떻게 되나요?"   # 변경 금액 680000원
EMAIL_SMALL = "제 항공권만 하루 뒤로 바꾸고 싶습니다. 3만 원 정도면 될까요?"   # 변경 금액 30000원
POLICY_BIG = "네 명 동시 변경이므로 수수료를 1인분만 받는다. 변경 희망 날짜의 네 좌석을 함께 확인해 준다."   # 고액 건에 담당자가 주는 방침 (대본)


### 단계 ① — 상태 정의 (요구사항 1)

그래프가 도는 동안 모든 노드가 함께 읽고 쓰는 키를 선언합니다. `messages` 키에 붙인 `add_messages`가 리듀서(reducer)입니다. 노드가 `messages`에 메시지를 돌려주면 리듀서가 기존 목록 뒤에 붙이고, `RemoveMessage`를 돌려주면 같은 `id`의 메시지를 지웁니다.

In [ ]:
# 여기에 단계 ①(상태 정의, 키 다섯 개)을 작성합니다.

### 단계 ② — 노드 함수 정의 (요구사항 2, 3, 4)

- `THRESHOLD`는 담당자에게 묻는 기준 금액, `DEFAULT_POLICY`는 소액 건의 기본 방침입니다. 두 상수가 개입 여부를 정하는 원칙입니다.
- ask_policy만 `interrupt()`를 부릅니다. auto_policy는 멈추지 않습니다.

In [ ]:
# 여기에 단계 ②(원칙 상수와 노드 함수 다섯 개 정의)를 작성합니다.

### 단계 ③ — 그래프 빌더 생성과 노드 등록 (요구사항 5)

In [ ]:
# 여기에 단계 ③(그래프 빌더 생성과 노드 등록)을 작성합니다.

### 단계 ④ — 엣지 연결 (요구사항 6)

manage 뒤의 조건부 엣지가 `amount`를 보고 두 방침 노드 중 하나를 고릅니다. 두 분기는 draft에서 합류합니다.

In [ ]:
# 여기에 단계 ④(판단 함수, 고정 엣지와 조건부 엣지)를 작성합니다.

### 단계 ⑤ — 컴파일과 실행 (요구사항 7, 8)

체크포인터를 달아 컴파일하고, 같은 `thread_id`로 고액 문의와 소액 문의를 차례로 실행합니다. 체크포인터 장착과 고액·소액 문의 두 통의 실행이 모두 이 한 단계에 속합니다.

실행 설정은 `config = {"configurable": {"thread_id": "customer-8102"}}`이고, 멈춘 지점은 `graph.get_state(config).next`, 담당자에게 간 내용은 `invoke` 반환값의 `out["__interrupt__"][0].value`에서 읽습니다.


In [ ]:
# 여기에 단계 ⑤(체크포인터 장착·컴파일, 고액 문의와 소액 문의의 실행)를 작성합니다.

## 7. 실행 결과 확인

위 실행 결과에서 다음 세 가지를 확인합니다.

1. 1번 메일(고액)에서 `next = ('ask_policy',)`가 출력되고, 담당자에게 간 내용에 변경 금액이 들어 있습니다. `[초안]`과 `[발송]`은 `Command(resume=…)` 뒤에만 출력됩니다.
2. 2번 메일(소액)에서 `__interrupt__ 있음 = False`가 출력되고, `[방침]` 줄이 기본 방침 문장이며, `Command(resume=…)` 없이 `[발송]`까지 진행됩니다.
3. 두 메일 모두 `[초안]`이 `[방침]`의 핵심을 담고, 2번 메일의 `[대화 기록]`에 1번 메일과 그 답장이 남아 있습니다.
